# AG_PRAXIS NB08c — Observation Budgets Read as Elapsed Time

NB08 measured how many records of a class have to be observed before detection stops
improving with further observation, and reported those budgets in records. A record is not
a unit anyone outside this project thinks in. This notebook converts the same budgets into
elapsed time, using the IAT column carried in the test records themselves.

Nothing is trained here, nothing is tuned, and no hypothesis is tested. H2 is measured on
saturation counted in records, and that measurement is untouched by anything below. What
this produces is a second way of expressing budgets that were already measured, decided
after they were measured, and it stands or falls on its own.

Two things have to be settled before any time figure means anything, and both are handled
below rather than assumed. What unit the IAT column is in, which the dataset does not
document. And how many packets one record summarises, which this project has not resolved.
The second is why every figure below is given twice.

The usual first cell: Drive, the repository, and the commit this ran at.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Every input is resolved and checked here, before anything is read. There are three of
them. The saved test records, which carry the IAT and Rate columns everything below works
from. The NB04 manifest, which holds the scaler's mean and scale for every column by name
and the order the kept columns sit in. And the NB08 saturation table, which holds the
per-class budgets being converted.

The records-to-threshold table is looked for in the same directory. If it is there it is
converted on the same two readings. If it is not, the analysis is restricted to saturation
and the file this writes records that restriction rather than leaving a reader to infer it
from an absence.

A missing input stops the run here. A notebook that gets halfway through and then fails on
a path has already spent the session.

In [ ]:
import json

import numpy as np
import pandas as pd

from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)
SEED = CFG["seed"]
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = ARTIFACTS / ("NB08c_fast" if FAST else "NB08c")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(f"{ARTIFACTS} does not exist, so nothing this notebook writes survives.")


def first_existing(candidates, what):
    """The first candidate that is there, or a stop with every path that was tried."""
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


RECORDS_NPZ = first_existing(
    [REPO_ROOT / "data" / "processed" / "records_test.npz",
     ARTIFACTS / "NB04" / "records_test.npz"], "the saved test records")
MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"],
    "the manifest holding the scaler and the kept column order")
TABLES_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB08" / "tables",
     ARTIFACTS / "NB08" / "tables"], "the directory NB08 wrote its tables to")

SATURATION_CSV = TABLES_DIR / "saturation.csv"
THRESHOLD_CSV = TABLES_DIR / "records_to_threshold.csv"
THRESHOLD_PRESENT = THRESHOLD_CSV.exists()
RESTRICTED_TO_SATURATION = not THRESHOLD_PRESENT

INPUTS = {
    "test records": RECORDS_NPZ,
    "manifest": MANIFEST_PATH,
    "saturation table": SATURATION_CSV,
}
if THRESHOLD_PRESENT:
    INPUTS["records-to-threshold table"] = THRESHOLD_CSV

ABSENT = {name: str(path) for name, path in INPUTS.items() if not path.exists()}
if ABSENT:
    raise FileNotFoundError(f"these inputs are not where they were looked for: {ABSENT}")

for name, path in INPUTS.items():
    print(f"{name:<26} {path}")
print()
print("records to threshold      "
      + ("found, and converted on both readings below" if THRESHOLD_PRESENT else
         f"not found under {TABLES_DIR}; the analysis is restricted to saturation"))
print()
print(f"writing to                {OUT_DIR}")
print("the copy that goes into the repository afterwards is "
      "data/processed/NB08c/observation_time.json")

Seeding, and what the session is running on. Nothing here is trained, but one seeded choice
is made below, the 100,000-row sample the unit evidence is measured over, so the seed
matters and it is taken from the config rather than written into the cell that uses it. The
versions go into what this writes for the reason they do everywhere else in the project: a
run whose environment is remembered rather than recorded cannot be read back later against
the machine that produced it.

In [ ]:
import random

import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    named = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        named.append(str(details.get("device_name", device.name)))
    return ", ".join(named)


ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}

CHECKS = []


def confirm(name, ok, detail):
    """Record a check and its outcome, and stop the run if it failed.

    Every check this notebook makes goes through here, so the file it writes carries the
    list of what was verified rather than a claim that verification happened.
    """
    CHECKS.append({"check": name, "outcome": "pass" if ok else "fail", "detail": str(detail)})
    if not ok:
        raise AssertionError(f"{name}: {detail}")
    return ok


print(f"seeded with : {SEED}")
print(f"environment : {ENVIRONMENT}")

Three things to hold on to before any number appears, because they bound what every figure
below can be used for.

The dataset has no timestamp column. Nothing here is a measured elapsed time. It is a
reconstruction, a count of records multiplied by a feature that describes inter-arrival
time, and it inherits whatever that feature does and does not represent.

IAT is one of the four timing features NB03 measured carrying recording provenance. A model
given that family alone identifies which of fifty recordings a row came from far above
chance, which is why the models the explanations are computed on have the family removed.
The figures below therefore describe these recordings as they were captured. They do not
transfer to another capture of the same attack under other conditions.

How many packets one record summarises is not resolved in this project. Section 4 of
`PROJECT_RECORD.md` records that the dataset's authors extracted features over 100-packet
windows for DDoS and DoS traffic and over 10-packet windows for everything else, and
Section 11 records that reading the Number column back through the scaler did not reproduce
that division. The two cannot both describe what a row summarises. Neither is discarded
here, so every time figure is given twice, once ignoring the extraction windows and once
applying them.

What is fixed here, before anything is measured.

The two groups are the ones Section 3 of `PROJECT_RECORD.md` fixes from the benchmark
taxonomy rather than from anything measured in this project. Volumetric is the eight DDoS
and DoS classes. Low-rate is Recon-OS_Scan, Recon-Port_Scan, Recon-VulScan, Spoofing and
MQTT-Malformed_Data. Recon-Ping_Sweep is excluded from the low-rate median under
`PREREGISTRATION.md` Amendment 4, which excluded it because it yields four test sequences
and no per-class metric on four samples is interpretable. The low-rate listing above already omits it,
so applying the exclusion by name takes nothing out of it. It is applied by name anyway, and
how many classes it removed is recorded in the file this writes, because a median that
depends on an exclusion being carried in a group listing somewhere else is a median that
changes silently if the listing is ever edited.

The extraction windows are the ones Section 4 records: 100 packets for the eight DDoS and
DoS classes, 10 packets for every other class. The eight are named rather than matched on
their names, because five other class names carry DoS or DDoS inside them, belong to the
MQTT family, and sit in the 10-packet remainder. Matching on the string would move them
into the 100-packet group without saying so.

Reading A multiplies the budget in records by the class's median IAT and stops there.
Reading B multiplies that product by the packets the class's extraction window holds. Both
are reported for every class and neither is preferred, because which one is right depends
on the open item in Section 11 and that item is open.

In [ ]:
VOLUMETRIC = ["DDoS-ICMP", "DDoS-SYN", "DDoS-TCP", "DDoS-UDP",
              "DoS-ICMP", "DoS-SYN", "DoS-TCP", "DoS-UDP"]
LOW_RATE = ["Recon-OS_Scan", "Recon-Port_Scan", "Recon-VulScan", "Spoofing",
            "MQTT-Malformed_Data"]
EXCLUDED_FROM_LOW_RATE = ["Recon-Ping_Sweep"]
EXCLUSION_BASIS = ("PREREGISTRATION.md Amendment 4: 4 test sequences at window 50 and "
                   "stride 25, too few for a per-class metric to be interpretable")

# The 100-packet set is the eight named above and nothing else. The MQTT flood classes
# carry DoS and DDoS in their names and are in the 10-packet remainder under Section 4.
WINDOW_PACKETS = 100
OTHER_PACKETS = 10
PACKETS_FOR = {}

READING_A = "the saturation budget in records multiplied by the class's median raw IAT"
READING_B = (f"the same product multiplied by the packets the class's extraction window "
             f"holds, {WINDOW_PACKETS} for the eight DDoS and DoS classes and "
             f"{OTHER_PACKETS} for every other class, following PROJECT_RECORD.md Section 4")
READINGS_STATUS = ("both are reported and neither is preferred; which one describes a row "
                   "depends on the packets-per-row item open in PROJECT_RECORD.md Section 11")

LOW_RATE_USED = [name for name in LOW_RATE if name not in EXCLUDED_FROM_LOW_RATE]
REMOVED_BY_EXCLUSION = [name for name in LOW_RATE if name in EXCLUDED_FROM_LOW_RATE]

print("volumetric group, and the 100-packet extraction window")
for name in VOLUMETRIC:
    print(f"  {name}")
print()
print("low-rate group, and the 10-packet extraction window")
for name in LOW_RATE:
    print(f"  {name}")
print()
print(f"excluded from the low-rate median: {', '.join(EXCLUDED_FROM_LOW_RATE)}")
print(f"  because {EXCLUSION_BASIS}")
print(f"  applying it removes {len(REMOVED_BY_EXCLUSION)} of the classes listed above, "
      "which already omits it; it is applied by name so the median cannot pick the class "
      "up if that listing ever changes")
print()
print(f"reading A: {READING_A}")
print(f"reading B: {READING_B}")

The records are stored standardised, so IAT and Rate have to be put back on their own scale
before either can be read as a time or a rate. The manifest holds the mean and the scale the
NB04 scaler fitted for every column by name, and the order the kept columns sit in, so the
recovery is the column's index out of the kept list and then that column's own two numbers.

The column count is asserted rather than taken on trust. If the array does not hold the 44
columns the kept list names then the index of IAT is not the IAT column, and every figure
below would be computed on some other feature without anything failing.

Which class each record belongs to is read from the array's own class list, and the
per-class counts are then checked against the counts the manifest recorded when the array
was written. If those disagree then the codes do not mean what the names say and every
per-class row below would be mislabelled.

In [ ]:
MANIFEST = json.loads(MANIFEST_PATH.read_text())
KEPT = [str(name) for name in MANIFEST["columns"]["kept"]]
SCALER_MEAN = MANIFEST["scaler"]["mean"]
SCALER_SCALE = MANIFEST["scaler"]["scale"]
MANIFEST_BY_CLASS = {str(k): int(v) for k, v in
                     MANIFEST["arrays"]["records_test"]["by_class"].items()}

with np.load(RECORDS_NPZ, allow_pickle=False) as npz:
    X = npz["X"]
    Y = npz["y"].astype("int64")
    CLASSES = ([str(v) for v in npz["classes"]] if "classes" in npz.files
               else sorted(MANIFEST_BY_CLASS))

N_RECORDS = int(len(Y))

assert X.shape[1] == 44, "the record array does not hold the 44 columns the kept list names"

confirm("the manifest's kept list names 44 columns", len(KEPT) == 44, f"{len(KEPT)} names")
confirm("the record array has one label for every row", X.shape[0] == len(Y),
        f"{X.shape[0]:,} rows and {len(Y):,} labels")
confirm("IAT and Rate are both in the kept list",
        "IAT" in KEPT and "Rate" in KEPT,
        f"IAT at index {KEPT.index('IAT') if 'IAT' in KEPT else 'absent'}, "
        f"Rate at index {KEPT.index('Rate') if 'Rate' in KEPT else 'absent'}")
confirm("the scaler carries a mean and a scale for both, and neither scale is zero",
        all(name in SCALER_MEAN and float(SCALER_SCALE.get(name, 0)) > 0
            for name in ("IAT", "Rate")),
        f"IAT mean {SCALER_MEAN.get('IAT')} scale {SCALER_SCALE.get('IAT')}, "
        f"Rate mean {SCALER_MEAN.get('Rate')} scale {SCALER_SCALE.get('Rate')}")


def raw(column):
    """Undo the standardization for one named column, with the manifest's own two numbers."""
    at = KEPT.index(column)
    return X[:, at].astype("float64") * float(SCALER_SCALE[column]) + float(SCALER_MEAN[column])


IAT = raw("IAT")
RATE = raw("Rate")

NAME_OF = np.asarray(CLASSES, dtype=object)
confirm("every label code in the array names one of the classes the array lists",
        int(Y.min()) >= 0 and int(Y.max()) < len(CLASSES),
        f"codes {int(Y.min())} to {int(Y.max())} against {len(CLASSES)} names")

RECORD_CLASS = NAME_OF[Y]
COUNTED = {name: int((RECORD_CLASS == name).sum()) for name in CLASSES}
DISAGREEING = {name: (COUNTED[name], MANIFEST_BY_CLASS.get(name))
               for name in CLASSES if COUNTED[name] != MANIFEST_BY_CLASS.get(name)}
confirm("the per-class counts of the array agree with the counts the manifest recorded",
        not DISAGREEING,
        "every class agrees" if not DISAGREEING else str(DISAGREEING))

print(f"{N_RECORDS:,} test records over {len(CLASSES)} classes")
print(f"IAT  raw: min {IAT.min():.6g}, median {np.median(IAT):.6g}, max {IAT.max():.6g}")
print(f"Rate raw: min {RATE.min():.6g}, median {np.median(RATE):.6g}, max {RATE.max():.6g}")

The rule for the unit, written down before the measurement it governs.

The dataset does not document what unit IAT is in, and nothing here may assume one. What
can be measured is how IAT and Rate sit against each other. If Rate is packets per second
and IAT is a mean gap between packets expressed in units of U seconds, then the two are
reciprocals up to that unit and their per-record product is 1/U. A product sitting at
10^6 says U is a microsecond. A product at 10^3 says a millisecond.

So the measurement is the product of raw IAT and raw Rate, per record, over a fixed sample
of 100,000 rows drawn at seed 42, reported as quantiles.

The rule, fixed now. If at least 90 percent of the sampled products fall within half a
decade of the median's power of ten, meaning within a factor of about three either way,
then the product concentrates on one power of ten. That power is read as 1/U, the implied
unit is stated, and every time figure below is carried in seconds. Products that are not
positive cannot be placed on a log scale at all, so they count against the 90 percent
rather than being dropped out of it.

If the product does not concentrate then no unit is implied and none is invented. Every
time figure below is then reported in IAT units, the unit is recorded as unresolved in the
file this writes, and nothing from this notebook reaches a chapter until the unit is
settled by something other than this notebook.

In [ ]:
SAMPLE_ROWS = 100_000
SAMPLE_SEED = 42
CONCENTRATION_RULE = ("at least 90 percent of the sampled products within half a decade of "
                      "the median's power of ten; non-positive products count against it")
CONCENTRATION_MARK = 0.90

SAMPLE_AT = np.random.default_rng(SAMPLE_SEED).choice(
    N_RECORDS, size=int(min(SAMPLE_ROWS, N_RECORDS)), replace=False)
PRODUCT = IAT[SAMPLE_AT] * RATE[SAMPLE_AT]
POSITIVE = PRODUCT > 0
N_NON_POSITIVE = int((~POSITIVE).sum())

QUANTILE_AT = [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]
QUANTILES = {f"{q:g}": float(np.quantile(PRODUCT, q)) for q in QUANTILE_AT}

if int(POSITIVE.sum()) == 0:
    DECADE = None
    WITHIN_DECADE = 0.0
else:
    LOG_PRODUCT = np.log10(PRODUCT[POSITIVE])
    DECADE = int(np.round(float(np.median(LOG_PRODUCT))))
    WITHIN_DECADE = float((np.abs(LOG_PRODUCT - DECADE) <= 0.5).sum() / len(PRODUCT))

CONCENTRATED = bool(DECADE is not None and WITHIN_DECADE >= CONCENTRATION_MARK)

NAMED_UNIT = {0: "seconds", 3: "milliseconds", 6: "microseconds", 9: "nanoseconds"}
if CONCENTRATED:
    SECONDS_PER_IAT_UNIT = float(10.0 ** (-DECADE))
    IAT_UNIT = NAMED_UNIT.get(
        DECADE, f"{SECONDS_PER_IAT_UNIT:.0e} seconds, which is not a named unit")
    TIME_UNIT = "seconds"
else:
    # Unresolved means the figures stay on the column's own scale. The factor is 1.0 so no
    # arithmetic below has to branch on it, and the unit travels as a name instead.
    SECONDS_PER_IAT_UNIT = 1.0
    IAT_UNIT = "unresolved"
    TIME_UNIT = "IAT units"

UNIT_EVIDENCE = {
    "rule": CONCENTRATION_RULE,
    "rule_fixed_before_the_measurement": True,
    "sample_rows": int(len(SAMPLE_AT)),
    "sample_seed": SAMPLE_SEED,
    "quantiles_of_iat_times_rate": QUANTILES,
    "non_positive_products": N_NON_POSITIVE,
    "median_power_of_ten": DECADE,
    "share_within_half_a_decade": WITHIN_DECADE,
    "concentration_mark": CONCENTRATION_MARK,
    "concentrates": CONCENTRATED,
    "implied_iat_unit": IAT_UNIT,
    "seconds_per_iat_unit": SECONDS_PER_IAT_UNIT if CONCENTRATED else None,
    "time_figures_reported_in": TIME_UNIT,
}

print(f"product of raw IAT and raw Rate over {len(SAMPLE_AT):,} rows at seed {SAMPLE_SEED}")
for q, value in QUANTILES.items():
    print(f"  quantile {q:<6} {value: .6g}")
print(f"  not positive  {N_NON_POSITIVE:,} of {len(SAMPLE_AT):,}")
print()
if CONCENTRATED:
    print(f"the products concentrate: {WITHIN_DECADE:.4%} sit within half a decade of "
          f"10^{DECADE}, against a mark of {CONCENTRATION_MARK:.0%}")
    print(f"read as 1/U, that implies IAT is in {IAT_UNIT} "
          f"({SECONDS_PER_IAT_UNIT:.0e} seconds per IAT unit)")
    print(f"every time figure below is in {TIME_UNIT}")
else:
    print(f"the products do not concentrate: {WITHIN_DECADE:.4%} sit within half a decade "
          f"of 10^{DECADE}, against a mark of {CONCENTRATION_MARK:.0%}")
    print("no unit is implied and none is invented")
    print(f"every time figure below is in {TIME_UNIT}, and the unit is recorded as "
          "unresolved")
    print("nothing from this notebook reaches a chapter until the unit is settled "
          "elsewhere")

The per-class IAT figures the conversion runs on. The median and the mean are both reported
with the record count behind them. The median is what the conversion uses, because a mean
over a long-tailed column is pulled up by a small number of large gaps and would make a
class look slower to observe than most of its records are. Where the two are far apart the
class's records are not one population and the single figure for that class says less about
it than the same figure says about a class where they agree.

In [ ]:
# Labels stay object dtype. A category dtype survives a groupby into the result and then
# raises on the arithmetic the readings below do.
PER_RECORD = pd.DataFrame({"class": pd.Series(RECORD_CLASS, dtype="object"),
                           "iat": pd.Series(IAT, dtype="float64")})

PER_CLASS = (PER_RECORD.groupby("class", sort=True)["iat"]
             .agg(iat_median_raw="median", iat_mean_raw="mean", test_records="count")
             .reset_index())
PER_CLASS["test_records"] = PER_CLASS["test_records"].astype("int64")

confirm("every numeric column of the per-class table is numeric",
        all(pd.api.types.is_numeric_dtype(PER_CLASS[column])
            for column in ("iat_median_raw", "iat_mean_raw", "test_records")),
        str({c: str(PER_CLASS[c].dtype) for c in PER_CLASS.columns}))
confirm("the per-class table holds one row for every class in the records",
        len(PER_CLASS) == len(CLASSES), f"{len(PER_CLASS)} rows against {len(CLASSES)} classes")
confirm("every class median is positive, so a budget times a median is a time",
        bool((PER_CLASS["iat_median_raw"] > 0).all()),
        f"smallest median {float(PER_CLASS['iat_median_raw'].min()):.6g}")

MEDIAN_IAT = dict(zip(PER_CLASS["class"], PER_CLASS["iat_median_raw"]))

print("raw IAT per class, on the column's own scale"
      + (f", which the measurement above read as {IAT_UNIT}" if CONCENTRATED
         else "; the unit is unresolved"))
print(PER_CLASS.to_string(index=False, float_format=lambda v: f"{v:.6g}"))

The saturation budgets, converted. The budget is the `saturates_at_k` column of the NB08
table, the smallest grid point at which the class's F1 sits within epsilon of its own
ceiling. The grid is 5, 10, 25 and 50, so a budget can only take one of those four values
and everything derived from one inherits that resolution.

Reading A is the budget times the class's median IAT. Reading B is that product times the
packets the class's extraction window holds, 100 for the eight DDoS and DoS classes and 10
for the rest. The two differ by a factor of ten between the groups, which is the size of
the thing Section 11 has not settled.

In [ ]:
SATURATION = pd.read_csv(SATURATION_CSV)

confirm("the saturation table carries the class and the budget columns this reads",
        {"class", "saturates_at_k"} <= set(SATURATION.columns),
        f"columns: {list(SATURATION.columns)}")
SATURATION["class"] = SATURATION["class"].astype("object")
confirm("every class in the saturation table is one of the classes in the test records",
        set(SATURATION["class"]) <= set(CLASSES),
        f"{len(SATURATION)} rows, "
        f"unmatched {sorted(set(SATURATION['class']) - set(CLASSES))}")

for name in CLASSES:
    PACKETS_FOR[name] = WINDOW_PACKETS if name in VOLUMETRIC else OTHER_PACKETS
confirm("exactly eight classes carry the 100-packet extraction window",
        sum(1 for v in PACKETS_FOR.values() if v == WINDOW_PACKETS) == 8,
        f"{sorted(n for n, v in PACKETS_FOR.items() if v == WINDOW_PACKETS)}")


def readings(frame, budget_column):
    """Both readings of a budget table, per class, on the medians measured above."""
    out = frame.loc[:, ["class", budget_column]].copy()
    out["budget_records"] = out[budget_column].astype("float64")
    out["packets_per_record"] = [PACKETS_FOR[name] for name in out["class"]]
    out["median_iat_raw"] = [MEDIAN_IAT[name] for name in out["class"]]
    out["reading_a"] = out["budget_records"] * out["median_iat_raw"] * SECONDS_PER_IAT_UNIT
    out["reading_b"] = out["reading_a"] * out["packets_per_record"]
    out["group"] = ["volumetric" if name in VOLUMETRIC else
                    "low-rate" if name in LOW_RATE else "neither" for name in out["class"]]
    return out.sort_values("class").reset_index(drop=True)


SATURATION_TIME = readings(SATURATION, "saturates_at_k")
SATURATION_TIME = SATURATION_TIME.drop(columns=["saturates_at_k"])

confirm("every reading is a finite positive number",
        bool(np.isfinite(SATURATION_TIME[["reading_a", "reading_b"]].to_numpy()).all())
        and bool((SATURATION_TIME[["reading_a", "reading_b"]].to_numpy() > 0).all()),
        f"{len(SATURATION_TIME)} classes, both readings finite and positive")
confirm("reading B is reading A times the class's packets per record",
        bool(np.allclose(SATURATION_TIME["reading_b"],
                         SATURATION_TIME["reading_a"] * SATURATION_TIME["packets_per_record"])),
        "the two readings differ only by the extraction window")

print(f"time to saturation per class, in {TIME_UNIT}")
print(SATURATION_TIME.to_string(index=False, float_format=lambda v: f"{v:.6g}"))

The group comparison, on each reading separately.

This is the median of the per-class times in each group, low-rate against volumetric, with
Recon-Ping_Sweep excluded from the low-rate side. What is reported is the direction of the
difference and nothing else. No ratio is computed and no pass mark is applied, because H2
states neither, and because H2 is measured on the budgets in records rather than on
anything here.

If both readings point the same way then the direction survives the unresolved packet
count. If they point opposite ways then the direction depends on it, and what is printed
says which reading gives the longer median to which group rather than choosing between
them.

In [ ]:
def direction_of(low, volumetric):
    """Which group takes longer, said in words rather than as a ratio."""
    if low > volumetric:
        return "low-rate classes take longer than volumetric classes"
    if volumetric > low:
        return "volumetric classes take longer than low-rate classes"
    return "the two group medians are equal"


def compare(frame, column):
    """The two group medians on one reading, and the direction between them."""
    low = frame.loc[frame["class"].isin(LOW_RATE_USED), column]
    volumetric = frame.loc[frame["class"].isin(VOLUMETRIC), column]
    low_median, volumetric_median = float(np.median(low)), float(np.median(volumetric))
    return {
        "low_rate_classes": LOW_RATE_USED,
        "volumetric_classes": VOLUMETRIC,
        "low_rate_median": low_median,
        "volumetric_median": volumetric_median,
        "difference_low_minus_volumetric": low_median - volumetric_median,
        "direction": direction_of(low_median, volumetric_median),
        "longer": ("low-rate" if low_median > volumetric_median else
                   "volumetric" if volumetric_median > low_median else "neither"),
    }


confirm("the low-rate side of the comparison is the group minus the excluded class",
        set(LOW_RATE_USED) == set(LOW_RATE) - set(EXCLUDED_FROM_LOW_RATE)
        and not set(LOW_RATE_USED) & set(EXCLUDED_FROM_LOW_RATE),
        f"{LOW_RATE_USED}; the exclusion removed {len(REMOVED_BY_EXCLUSION)} class"
        f"{'' if len(REMOVED_BY_EXCLUSION) == 1 else 'es'} from the listing")
confirm("both groups are present in the saturation table",
        set(LOW_RATE_USED) <= set(SATURATION_TIME["class"])
        and set(VOLUMETRIC) <= set(SATURATION_TIME["class"]),
        f"{len(LOW_RATE_USED)} low-rate and {len(VOLUMETRIC)} volumetric classes found")

COMPARISON = {"reading_a": compare(SATURATION_TIME, "reading_a"),
              "reading_b": compare(SATURATION_TIME, "reading_b")}
AGREE = COMPARISON["reading_a"]["longer"] == COMPARISON["reading_b"]["longer"]
if AGREE:
    AGREEMENT = (f"the two readings agree: on both, {COMPARISON['reading_a']['direction']}, "
                 "so the direction does not depend on the unresolved packet count")
else:
    AGREEMENT = (f"the two readings disagree: reading A gives the longer median to the "
                 f"{COMPARISON['reading_a']['longer']} group and reading B to the "
                 f"{COMPARISON['reading_b']['longer']} group, so the direction depends on "
                 "the unresolved packet count and neither reading settles it")

for reading in ("reading_a", "reading_b"):
    found = COMPARISON[reading]
    print(f"reading {reading[-1].upper()}, in {TIME_UNIT}")
    print(f"  low-rate median    {found['low_rate_median']:.6g}   "
          f"over {', '.join(found['low_rate_classes'])}")
    print(f"  volumetric median  {found['volumetric_median']:.6g}   "
          f"over {len(found['volumetric_classes'])} classes")
    print(f"  direction          {found['direction']}")
    print()
print(AGREEMENT)
print()
print("no ratio is computed and no pass mark is applied; H2 states neither, and H2 is "
      "measured on saturation in records, which nothing here moves")

The other budget table, when it is there. Records to threshold is the number of records at
which a class reaches F1 0.80, and it converts on the same two readings.

It carries censoring that the saturation table does not. A class that has not reached 0.80
by 50 records is written as `> 50`, which is a lower bound and not a budget, so its converted
time is a lower bound too and is marked as one. Section 11 of `PROJECT_RECORD.md` records
that three of the five low-rate classes do not reach 0.80 within 50 records, which is why
the low-rate median on this table is not determinate and why H2 is measured on saturation
instead. No group median is computed on this table here, and what was censored is counted
and named below rather than left as a claim.

In [ ]:
import re

THRESHOLD_TIME = None
THRESHOLD_NOTE = (f"no records-to-threshold table under {TABLES_DIR}; the analysis is "
                  "restricted to saturation")

if THRESHOLD_PRESENT:
    THRESHOLD = pd.read_csv(THRESHOLD_CSV)
    confirm("the records-to-threshold table carries the class and the budget columns",
            {"class", "reported_as"} <= set(THRESHOLD.columns),
            f"columns: {list(THRESHOLD.columns)}")
    THRESHOLD["class"] = THRESHOLD["class"].astype("object")
    REPORTED = THRESHOLD["reported_as"].astype("str").str.strip()
    THRESHOLD["censored"] = REPORTED.str.startswith(">")
    THRESHOLD["budget"] = [float(re.sub(r"[^0-9.]", "", text)) for text in REPORTED]
    confirm("every reported budget parses to a number",
            bool(np.isfinite(THRESHOLD["budget"].to_numpy()).all()),
            f"{len(THRESHOLD)} rows, {int(THRESHOLD['censored'].sum())} of them censored")

    THRESHOLD_TIME = readings(THRESHOLD, "budget").drop(columns=["budget"])
    THRESHOLD_TIME["censored"] = [bool(v) for v in
                                  THRESHOLD.sort_values("class")["censored"].to_numpy()]
    confirm("the censored rows carried through to the converted table",
            int(THRESHOLD_TIME["censored"].sum()) == int(THRESHOLD["censored"].sum()),
            f"{int(THRESHOLD_TIME['censored'].sum())} lower bounds")

    CENSORED_LOW_RATE = sorted(THRESHOLD_TIME.loc[THRESHOLD_TIME["censored"]
                                                  & THRESHOLD_TIME["class"].isin(LOW_RATE_USED),
                                                  "class"])
    THRESHOLD_NOTE = (
        f"{int(THRESHOLD_TIME['censored'].sum())} of {len(THRESHOLD_TIME)} classes are "
        f"censored at their largest grid point, {len(CENSORED_LOW_RATE)} of them in the "
        f"low-rate group ({', '.join(CENSORED_LOW_RATE) or 'none'}), so the low-rate median "
        "on this table is not determinate and no group comparison is computed on it")

    print(f"time to threshold per class, in {TIME_UNIT}; a censored row is a lower bound")
    print(THRESHOLD_TIME.to_string(index=False, float_format=lambda v: f"{v:.6g}"))
    print()

print(THRESHOLD_NOTE)

Writing it down, so every figure above can be read again without running any of this. The
file carries the unit evidence and the verdict on it, the per-class medians and means, both
readings for every class, both group comparisons, the exclusion that was applied and what
it rests on, and the commit this ran at.

In [ ]:
DOCUMENT = {
    "generated_by": "AG_PRAXIS_NB08c_observation_time.ipynb",
    "generated_on": RUN_DATE, "git_sha": GIT_SHA, "git_dirty": GIT_DIRTY,
    "seed": SEED, "is_fast_pass": bool(FAST), "environment": ENVIRONMENT,
    "trained": "nothing",
    "tests_no_hypothesis": True,
    "status": ("a post-hoc re-expression of observation budgets already measured in "
               "records; H2 is measured on saturation in records and nothing here moves it "
               "or its comparator"),
    "read": {name: str(path) for name, path in INPUTS.items()},
    "restricted_to_saturation": RESTRICTED_TO_SATURATION,
    "n_test_records": N_RECORDS,
    "classes": list(CLASSES),
    "caveats": [
        "the dataset has no timestamp column, so this is a reconstruction from a feature "
        "and not a measured elapsed time",
        "IAT is one of the four timing features NB03 measured carrying recording "
        "provenance, so these figures describe the recordings as captured and may not "
        "transfer to other capture conditions",
        "the per-row packet count is unresolved in PROJECT_RECORD.md Section 11, which is "
        "why both readings are reported and neither is preferred",
    ],
    "unit_evidence": UNIT_EVIDENCE,
    "groups": {
        "volumetric": VOLUMETRIC,
        "low_rate": LOW_RATE,
        "low_rate_used_in_the_median": LOW_RATE_USED,
        "fixed_by": "PROJECT_RECORD.md Section 3, from the benchmark taxonomy",
    },
    "exclusion": {
        "class": EXCLUDED_FROM_LOW_RATE,
        "applied_to": "the low-rate median on both readings",
        "basis": EXCLUSION_BASIS,
        "removed_from_the_listing": REMOVED_BY_EXCLUSION,
        "note": ("PROJECT_RECORD.md Section 3's low-rate listing already omits the class, "
                 "so applying the exclusion by name removes nothing from it; it is applied "
                 "by name so the median does not depend on that listing carrying it"),
    },
    "extraction_windows": {
        "packets_per_record": PACKETS_FOR,
        "source": "PROJECT_RECORD.md Section 4, from Doménech et al. (2025) Section 5.1.1",
        "open_item": "PROJECT_RECORD.md Section 11 records that reading the Number column "
                     "back through the scaler did not reproduce this division",
    },
    "readings": {"a": READING_A, "b": READING_B, "status": READINGS_STATUS,
                 "time_unit": TIME_UNIT},
    "per_class_iat": PER_CLASS.to_dict(orient="records"),
    "saturation": SATURATION_TIME.to_dict(orient="records"),
    "records_to_threshold": (None if THRESHOLD_TIME is None
                             else THRESHOLD_TIME.to_dict(orient="records")),
    "records_to_threshold_note": THRESHOLD_NOTE,
    "group_comparison": {
        **COMPARISON,
        "readings_agree": AGREE,
        "statement": AGREEMENT,
        "no_ratio": "no ratio and no pass mark; H2 states neither",
    },
    "assertions": CHECKS,
}

ARTIFACT = OUT_DIR / "observation_time.json"
ARTIFACT.write_text(json.dumps(DOCUMENT, indent=1))
print(f"wrote {ARTIFACT}")
print(f"{len(CHECKS)} checks recorded, all passing, or this cell would not have been reached")

The ledger entry for this run.

In [ ]:
status = ("DO NOT ENTER, fast pass" if FAST else
          "reported result, working tree dirty" if GIT_DIRTY else
          "reported re-expression; no model trained and no hypothesis tested")

ledger = f"""
### NB08c — observation budgets read as elapsed time ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB08c_observation_time.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| status | {status} |
| environment | {ENVIRONMENT} |
| trained | nothing; reads the saved test records, the NB04 scaler and the NB08 budget tables |
| read over | {N_RECORDS:,} test records, {len(CLASSES)} classes |
| IAT unit | {IAT_UNIT}; {UNIT_EVIDENCE['share_within_half_a_decade']:.4%} of {UNIT_EVIDENCE['sample_rows']:,} sampled products within half a decade of 10^{UNIT_EVIDENCE['median_power_of_ten']}, mark {CONCENTRATION_MARK:.0%} |
| time figures in | {TIME_UNIT} |
| reading A | {READING_A} |
| reading B | {READING_B} |
| low-rate median, reading A | {COMPARISON['reading_a']['low_rate_median']:.6g} |
| volumetric median, reading A | {COMPARISON['reading_a']['volumetric_median']:.6g} |
| direction, reading A | {COMPARISON['reading_a']['direction']} |
| low-rate median, reading B | {COMPARISON['reading_b']['low_rate_median']:.6g} |
| volumetric median, reading B | {COMPARISON['reading_b']['volumetric_median']:.6g} |
| direction, reading B | {COMPARISON['reading_b']['direction']} |
| the two readings | {AGREEMENT} |
| exclusion | {', '.join(EXCLUDED_FROM_LOW_RATE)} from the low-rate median, under PREREGISTRATION.md Amendment 4 |
| records to threshold | {THRESHOLD_NOTE} |
| artefact | {ARTIFACT} |
| hypothesis | none; a post-hoc re-expression of RO2's budgets, and H2 does not move |
"""

print(ledger)